# Step 1 — Prepare Data

Merge 4 translation block CSVs, **drop German**, clean text, and save:
`data/sentences.csv` with columns: `id | English | Spanish | French | Italian`

In [4]:
import os, glob, re
import numpy as np
import pandas as pd

BASE_DIR = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\UGSC_multilingual_sentiment_package\UGSC_multilingual_sentiment_package"
TRANSLATION_DIR = os.path.join(BASE_DIR, "translated_versions")
BLOCK_FILES = sorted(glob.glob(os.path.join(TRANSLATION_DIR, "block_*.csv")))

OUTPUT_PATH = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\data\sentences.csv"

ALL_LANGS  = ["English", "Spanish", "French", "German", "Italian"]
KEEP_LANGS = ["English", "Spanish", "French", "Italian"]  # German dropped

print(f"Found {len(BLOCK_FILES)} block files:")
for f in BLOCK_FILES:
    print(" ", os.path.basename(f))

Found 4 block files:
  block_1_translation_template.csv
  block_2_translation_template.csv
  block_3_translation_template.csv
  block_4_translation_template.csv


In [5]:
def clean_text(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = re.sub(r"[;,]+\s*$", "", val).strip()
    val = re.sub(r"\s{2,}", " ", val)
    return val if val else np.nan

# ── Load & merge ───────────────────────────────────────────────────────────────
frames = []
for path in BLOCK_FILES:
    # Real separator is ";" — the comma-separated header is just decoration
    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        header=None,
        names=ALL_LANGS,
        skiprows=1,        # skip the malformed header row
        on_bad_lines="skip",
        engine="python",
    )
    df = df[KEEP_LANGS]   # drop German here
    frames.append(df)

merged = pd.concat(frames, ignore_index=True)

# ── Clean ──────────────────────────────────────────────────────────────────────
for lang in KEEP_LANGS:
    merged[lang] = merged[lang].apply(clean_text)

merged = merged[merged["English"].notna()].reset_index(drop=True)

before = len(merged)
merged = merged.drop_duplicates(subset="English").reset_index(drop=True)
after  = len(merged)

merged.insert(0, "id", range(1, after + 1))

print(f"Rows after merge  : {before}")
print(f"Rows after dedup  : {after}  (removed {before-after} duplicates)")
print(f"\nMissing per language:")
print(merged[KEEP_LANGS].isna().sum())
merged.head(8)

Rows after merge  : 200
Rows after dedup  : 200  (removed 0 duplicates)

Missing per language:
English     0
Spanish    67
French      0
Italian     0
dtype: int64


,id,English,Spanish,French,Italian
0,1,Cable car gives quick access to spectacular vi...,El telefrico ofrece acceso rpido a vistas es...,Le tlphrique offre un accs rapide  des vu...,La funivia offre un rapido accesso a viste spe...
1,2,A museum at the top in an old fort and also go...,Un museo en la cima de un antiguo fuerte y tam...,Un muse au sommet d'un vieux fort et un bon r...,Un museo in cima in un vecchio forte e anche u...
2,3,Dubrovnik Old Town is magical and nowhere more...,"El casco antiguo de Dubrovnik es mgico, y en ...","La vieille ville de Dubrovnik est magique, et ...",La citt vecchia di Dubrovnik  magica e da ne...
3,4,Even more disgruntled that we did not even get...,ÁQu disgusto an mayor que ni siquiera recibi...,Nous sommes encore plus mcontents de ne pas a...,Ancora pi delusi dal fatto che non abbiamo ne...
4,5,A very short and expensive trip .,Un viaje muy corto y caro.,Un trajet trs court et cher.,Un viaggio molto breve e costoso.
5,6,Do not waste your money on the cable car .,No malgastes tu dinero en el telefrico.,Ne gaspillez pas votre argent dans le tlphr...,Non sprecare i tuoi soldi con la funivia.
6,7,A bargain and a must .,"Una ganga, una visita obligada.",Une bonne affaire et un incontournable.,Un affare e un must.
7,8,A very short ride for an expensive price of a ...,"Un viaje muy corto por un precio elevado, con ...","Un trajet trs court pour un prix lev, avec ...",Un viaggio molto breve a un prezzo elevato per...


In [6]:
complete = merged[KEEP_LANGS].notna().all(axis=1).sum()
print(f"Rows with all 4 languages complete: {complete} / {after}")

merged.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\nSaved → {OUTPUT_PATH}")
merged.tail(5)

Rows with all 4 languages complete: 133 / 200

Saved → c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\data\sentences.csv


,id,English,Spanish,French,Italian
195,196,( we only needed 3 taxis for 9 of us ) They re...,NaN,(Nous n'avons eu besoin que de 3 taxis pour 9 ...,(abbiamo avuto bisogno solo di 3 taxi per 9 di...
196,197,"Although they have an elevator , it does not g...",NaN,"Bien qu'ils disposent d'un ascenseur, il ne de...","Sebbene abbiano un ascensore, non va al livell..."
197,198,Do not miss the War Museum on MountSrd .,NaN,Ne manquez pas le muse de la guerre sur le mo...,Non perdetevi il Museo della Guerra sul Monte ...
198,199,But I do not find the top of the hill anything...,NaN,Mais je ne trouve pas le sommet de la colline ...,Ma non trovo la cima della collina niente di s...
199,200,Definitely the best views along the Adriatic c...,NaN,Sans aucun doute les plus belles vues de la c...,Sicuramente la vista migliore lungo la costa a...
